In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'[.…]+', '.', phonetic)

    phonetic = re.sub(r'^\.|\.$', '', phonetic)

    return phonetic

print(normalize_phonetic('….daj˧.….nɯŋ˨˩'))

daj˧.nɯŋ˨˩


In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

ValueError: Could not align 'การขัดกันของผลประโยชน์' with 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16678/18310 [04:39<00:17, 91.93it/s] c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:59<00:00, 61.10it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [ ]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [ ]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18262,ไอซ์แลนด์,ʔajs˦˥.lɛːn˧,1,ʔajs˦˥.lɛːn˧,None,ERROR: argument of type 'NoneType' is not iter...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,None,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,None,ERROR: argument of type 'NoneType' is not iter...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [ ]:
errors.sample(10)   

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
13309,อยู่ไฟ,juː˨˩.faj˧,1,juː˨˩.faj˧,None,ERROR: Could not align 'อยู่ไฟ' with 'juː˨˩.faj˧'
10668,วนิพก,wa˦˥.nip̚˦˥.pʰok̚˦˥,2,waʔ˦˥.nip˦˥.pʰok˦˥,None,ERROR: Could not align 'วนิพก' with 'waʔ˦˥.nip...
3989,ซอส,sɔːs˦˥,2,sɔːs˦˥,None,ERROR: argument of type 'NoneType' is not iter...
7590,ผบห.,pʰɔː˩˩˦.bɔː˧.hɔː˩˩˦,1,pʰɔː˨˥.bɔː˧.hɔː˨˥,None,ERROR: Could not align 'ผบห.' with 'pʰɔː˨˥.bɔː...
4370,ดิสนีย์,dis˦˥.niː˥˩,1,dis˦˥.niː˦˩,None,ERROR: argument of type 'NoneType' is not iter...
13564,อัลบั้ม,ʔal˧.bam˥˩,2,ʔal˧.bam˦˩,None,ERROR: argument of type 'NoneType' is not iter...
8919,มอดูล,mɔː˧.duːl˧,2,mɔː˧.duːl˧,None,ERROR: argument of type 'NoneType' is not iter...
16773,แพลเลเดียม,pʰɛːl˧.leː˧.dia̯m˥˩,2,pʰɛːl˧.leː˧.diəm˦˩,None,ERROR: argument of type 'NoneType' is not iter...
9098,มาตราการ,maːt̚˥˩.traː˧.kaːn˧,1,maːt˦˩.traː˧.kaːn˧,None,ERROR: Could not align 'มาตราการ' with 'maːt˦˩...
4212,ด,dɔː˧.dek̚˨˩,2,dɔː˧.dek˨˩,None,ERROR: Could not align 'ด' with 'dɔː˧.dek˨˩'


In [ ]:
mask = ~errors["normalized_phonetic"].str.contains(
    r"[lsf][˥˦˧˨˩]", regex=True, na=False
)
filtered_errors = errors[mask]
filtered_errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18045,ไปสู่สรวงสวรรค์,paj˧.suː˨˩.sua̯ŋ˩˩˦.sa˨˩.wan˩˩˦,1,paj˧.suː˨˩.suəŋ˨˥.saʔ˨˩.wan˨˥,None,ERROR: Could not align 'ไปสู่สรวงสวรรค์' with ...
18055,ไพฑูรย์,pʰaj˧.tʰuːn˧,1,pʰaj˧.tʰuːn˧,None,ERROR: Could not align 'ไพฑูรย์' with 'pʰaj˧.t...
18132,ไม้กอล์ฟ,maːj˦˥.kɔːp̚˦˥,2,maːj˦˥.kɔːp˦˥,None,ERROR: Could not align 'ไม้กอล์ฟ' with 'maːj˦˥...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [ ]:
filtered_errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
7048,ประธานาธิบดี,pra˨˩.tʰaː˧.naː˧.tʰi˦˥.bɔː˧.diː˧,2,praʔ˨˩.tʰaː˧.naː˧.tʰiʔ˦˥.bɔː˧.diː˧,None,ERROR: Could not align 'ประธานาธิบดี' with 'pr...
11376,สมณะกระทรวง,sa˨˩.ma˦˥.na˦˥.kra˨˩.sua̯ŋ˧,1,saʔ˨˩.maʔ˦˥.naʔ˦˥.kraʔ˨˩.suəŋ˧,None,ERROR: Could not align 'สมณะกระทรวง' with 'saʔ...
4149,ฌ,t͡ɕʰɔː˧,2,t͡ɕʰɔː˧,None,ERROR: Could not align 'ฌ' with 't͡ɕʰɔː˧'
13655,อาถรรพณ์,ʔaː˧.tʰan˩˩˦,1,ʔaː˧.tʰan˨˥,None,ERROR: Could not align 'อาถรรพณ์' with 'ʔaː˧.t...
3864,ชื่อผู้ใช้,t͡ɕʰɯː˥˩.pʰuː˥˩.t͡ɕʰaj˦˥,1,t͡ɕʰɯː˦˩.pʰuː˦˩.t͡ɕʰaj˦˥,None,ERROR: Could not align 'ชื่อผู้ใช้' with 't͡ɕʰ...
5176,ทรงยี่สิบหน้า,soŋ˧.jiː˥˩.sip̚˨˩.naː˥˩,1,soŋ˧.jiː˦˩.sip˨˩.naː˦˩,None,ERROR: Could not align 'ทรงยี่สิบหน้า' with 's...
9161,มิจฉาทิฐิ,mit̚˦˥.t͡ɕʰaː˩˩˦.tʰit̚˦˥.tʰiʔ˨˩,1,mit˦˥.t͡ɕʰaː˨˥.tʰit˦˥.tʰiʔ˨˩,None,ERROR: Could not align 'มิจฉาทิฐิ' with 'mit˦˥...
13570,อัลลอหฺ,ʔan˧.lɔʔ˦˥,2,ʔan˧.lɔʔ˦˥,None,ERROR: Could not align 'อัลลอหฺ' with 'ʔan˧.lɔ...
5209,ทราบฝ่าละอองธุลีพระบาท,saːp̚˥˩.faː˨˩.la˦˥.ʔɔːŋ˧.tʰu˦˥.liː˧.pʰra˦˥.baː...,1,saːp˦˩.faː˨˩.laʔ˦˥.ʔɔːŋ˧.tʰuʔ˦˥.liː˧.pʰraʔ˦˥.b...,None,ERROR: Could not align 'ทราบฝ่าละอองธุลีพระบาท...
7245,ปลาซิวปลาสร้อย,plaː˧.siw˧.plaː˧.sɔj˥˩,1,plaː˧.siw˧.plaː˧.sɔj˦˩,None,ERROR: Could not align 'ปลาซิวปลาสร้อย' with '...
